In [1]:
import subprocess
import pickle

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, rankdata
from scipy.spatial.distance import pdist, squareform

from tqdm.notebook import tqdm

In [2]:
SEED = 0
N_PERM = 10_000

In [3]:
base = "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/"
data_dict_path = "n16_data_dict.pkl"
structure_key_path = "n16_structure_key_dict.pkl"
readme_path = "n16_readme.pkl"
with open(base+data_dict_path, 'rb') as f:
    data_dict = pickle.load(f)
with open(base+structure_key_path, 'rb') as f:
    structure_key_dict = pickle.load(f)
with open(base+readme_path, 'rb') as f:
    readme = pickle.load(f)
print(readme)

QuIC EXACT EMBEDDINGS — EXHAUSTIVE CONNECTED CUBIC GRAPHS, n=14

Generated: 2026-07-09
Producer:  quic-cycle-correlation-n16 (Kaggle)
Author:    Luke Miller (UMKC)
Context:   Decodability study of the QuIC quantum graph embedding
           (companion to QCE 2026 paper, arXiv:2604.18841).

CONTENTS
--------
n16_data_dict.pkl
    dict keyed by graph index (0..508), one entry per graph in the
    complete enumeration of connected 3-regular graphs on 14 vertices
    (509 graphs, generated by nauty-geng -c -d3 -D3 14, order as emitted).
    Per-entry fields:
      graph6         : str   — canonical graph6 string (reconstruct graph via
                               nx.from_graph6_bytes(s.encode()); circuit via
                               QuIK_circuit(adj_mat) with config below)
      adj_mat        : (14,14) float array — adjacency matrix
      num_triangles  : int   — simple 3-cycles (nx.simple_cycles, each once)
      num_4_cycles   : int   — simple 4-cycles
      num_5_cycles   : int

In [4]:
keys = sorted(data_dict)
vectors       = np.vstack([data_dict[k]['exact_vector'] for k in keys])
num_triangles = np.array([data_dict[k]['num_triangles'] for k in keys])
num_4_cycles  = np.array([data_dict[k]['num_4_cycles']  for k in keys])
num_5_cycles  = np.array([data_dict[k]['num_5_cycles']  for k in keys])

In [5]:
rng = np.random.default_rng(SEED)
rand_scalars = rng.random(len(keys))

emb_dist     = pdist(vectors, metric='cityblock')
tri_dist     = pdist(num_triangles.reshape(-1, 1), metric='cityblock')
cycle_4_dist = pdist(num_4_cycles.reshape(-1, 1),  metric='cityblock')
cycle_5_dist = pdist(num_5_cycles.reshape(-1, 1),  metric='cityblock')
rand_dist    = pdist(rand_scalars.reshape(-1, 1),  metric='cityblock')

In [6]:
def mantel_p(emb_dist, values, rho_obs, n_perm=N_PERM, seed=SEED):
    """Identical Mantel-Spearman statistic, computed via pair relabeling.

    Per permutation sigma, pdist(values[sigma]) is a relabeling of the
    original pair distances: d_perm[(i,j)] = d[(sigma_i, sigma_j)] — so
    one gather from a precomputed integer distance matrix replaces pdist,
    and Spearman = Pearson(z-scored ranks) with the emb side ranked once.
    Target values are small integers, so tie-averaged ranks per
    permutation are a lookup table over distinct distance values.
    P-values are Monte Carlo draws from the same null as before but a
    different RNG stream — statistically equivalent, not bit-identical
    to the old implementation.
    """
    rng = np.random.default_rng(seed)
    m = len(values)

    er = rankdata(emb_dist)
    er = (er - er.mean()) / er.std()

    # integer-coded pairwise target distances, full matrix (int16)
    codes = rankdata(values, method='dense').astype(np.int16) - 1
    vals = np.sort(np.unique(values))
    Dv = np.abs(vals[:, None] - vals[None, :])          # value-level dists
    dcode = rankdata(Dv.ravel(), method='dense').reshape(Dv.shape) - 1
    M = dcode[codes[:, None], codes[None, :]].astype(np.int32)
    dist_vals = np.sort(np.unique(Dv))                  # code -> distance
    assert dcode.max() < np.iinfo(np.int32).max, (
     f'distance codes ({dcode.max()}) overflow int32')


    iu0, iu1 = np.triu_indices(m, k=1)
    d0 = M[iu0, iu1]                                    # unpermuted codes

    def rho_of(dcodes):
        cnt = np.bincount(dcodes, minlength=len(dist_vals))
        # tie-averaged ranks per code: cumulative counts
        cum = np.cumsum(cnt)
        lut = (cum - (cnt - 1) / 2.0)                   # average rank
        r = lut[dcodes]
        r = (r - r.mean()) / r.std()
        return float(er @ r) / len(er)

    # sanity: fast rho on the identity permutation must match rho_obs
    assert abs(rho_of(d0) - rho_obs) < 1e-6, (
        f'fast-path rho {rho_of(d0):.8f} != scipy rho {rho_obs:.8f}')

    hits = 0
    for _ in tqdm(range(n_perm), leave=False):
        sig = rng.permutation(m)
        hits += (rho_of(M[sig[iu0], sig[iu1]]) >= rho_obs)
    return (hits + 1) / (n_perm + 1)


CHECKPOINT = ("/kaggle/input/notebooks/lukemiller1987/"
                   "aaai-n16-e1-stratified-residue/n16_e1_global_partial.pkl")
with open(CHECKPOINT, 'rb') as f:
    results = pickle.load(f)
print(f'checkpoint loaded: {sorted(results)}')
for name, target_dist, target_vals in [
        ('triangles', tri_dist,     num_triangles),
        ('4-cycles',  cycle_4_dist, num_4_cycles),
        ('5-cycles',  cycle_5_dist, num_5_cycles),
        ('control',   rand_dist,    rand_scalars)]:
    if name in results:
         print(f'{name:>10}:  rho = {results[name]["rho"]:+.3f},  '
               f'Mantel p = {results[name]["p"]:.4f}   (from checkpoint)')
         continue
    rho, _ = spearmanr(emb_dist, target_dist)
    p = mantel_p(emb_dist, target_vals, rho)
    results[name] = {'rho': rho, 'p': p}
    with open('/kaggle/working/n16_e1_global_partial.pkl', 'wb') as f:
        pickle.dump(results, f)                          # checkpoint per target
    print(f'{name:>10}:  rho = {rho:+.3f},  Mantel p = {p:.4f}')

checkpoint loaded: ['4-cycles', '5-cycles', 'triangles']
 triangles:  rho = +0.900,  Mantel p = 0.0001   (from checkpoint)
  4-cycles:  rho = +0.303,  Mantel p = 0.0001   (from checkpoint)
  5-cycles:  rho = +0.079,  Mantel p = 0.0001   (from checkpoint)


  0%|          | 0/10000 [00:00<?, ?it/s]

   control:  rho = -0.007,  Mantel p = 0.9720


In [7]:
# E1 level 1: within fixed-triangle strata, do C4 and C5 organize the geometry?
# Each stratum is an independent graph set -> Fisher's method combines per-stratum
# Mantel p-values into one gate decision number per target.

from scipy.stats import chi2
from numpy.linalg import svd

MIN_STRATUM = 15          # >= 105 pairs; below this the Mantel is underpowered
N_PERM_STRATUM = 10_000

def effective_rank(X):
    var = svd(X - X.mean(axis=0), full_matrices=False, compute_uv=False) ** 2
    return var.sum() ** 2 / (var ** 2).sum()

def stratum_test(idx, target_vals):
    """Mantel rho + p for one target within one stratum (graph indices idx)."""
    ed = pdist(vectors[idx], metric='cityblock')
    td = pdist(target_vals[idx].reshape(-1, 1), metric='cityblock')
    if td.max() == 0:                       # target constant within stratum
        return None, None
    rho, _ = spearmanr(ed, td)
    p = mantel_p(ed, target_vals[idx], rho, n_perm=N_PERM_STRATUM)
    return rho, p

def fisher_combine(pvals):
    pvals = [p for p in pvals if p is not None]
    stat = -2 * np.sum(np.log(pvals))
    return chi2.sf(stat, df=2 * len(pvals)), len(pvals)

stratum_results = {}
for label, members in structure_key_dict.items():
    if not label.startswith('tri_') or len(members) < MIN_STRATUM:
        continue
    idx = np.array(sorted(members))
    rho4, p4 = stratum_test(idx, num_4_cycles)
    rho5, p5 = stratum_test(idx, num_5_cycles)
    er = effective_rank(vectors[idx])
    stratum_results[label] = {'n': len(idx), 'rho_C4': rho4, 'p_C4': p4,
                              'rho_C5': rho5, 'p_C5': p5, 'eff_rank': er}
    print(f'{label:>7} (n={len(idx):3d}):  '
          f'C4 rho={rho4:+.3f} p={p4:.4f}   '
          f'C5 rho={rho5:+.3f} p={p5:.4f}   '
          f'eff_rank={er:.2f}')

p_global_C4, k4 = fisher_combine([v['p_C4'] for v in stratum_results.values()])
p_global_C5, k5 = fisher_combine([v['p_C5'] for v in stratum_results.values()])
print(f'\nFisher combined across {k4} strata:  '
      f'C4 p = {p_global_C4:.2e},  C5 p = {p_global_C5:.2e}')

  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

  tri_0 (n=792):  C4 rho=+0.964 p=0.0001   C5 rho=+0.238 p=0.0001   eff_rank=2.99


  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

...

  0%|          | 0/10000 [00:00<?, ?it/s]

  tri_5 (n=101):  C4 rho=+0.883 p=0.0001   C5 rho=+0.312 p=0.0001   eff_rank=3.05


  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

  tri_6 (n= 37):  C4 rho=+0.853 p=0.0001   C5 rho=+0.503 p=0.0001   eff_rank=3.04

Fisher combined across 7 strata:  C4 p = 1.10e-20,  C5 p = 1.10e-20


In [8]:
# The organization question, not just correlation: inside the largest strata,
# which axis is PC1 now? If C4 aligns with the leading within-stratum PC,
# the hierarchy claim is direct.

big_strata = sorted(stratum_results, key=lambda l: -stratum_results[l]['n'])[:4]

for label in big_strata:
    idx = np.array(sorted(structure_key_dict[label]))
    Xc = vectors[idx] - vectors[idx].mean(axis=0)
    U, S, Vt = svd(Xc, full_matrices=False)
    var_ratio = S**2 / (S**2).sum()
    scores = Xc @ Vt.T
    print(f'\n{label} (n={len(idx)}):')
    for pc in range(3):
        a4 = spearmanr(scores[:, pc], num_4_cycles[idx]).statistic
        a5 = spearmanr(scores[:, pc], num_5_cycles[idx]).statistic
        print(f'  PC{pc+1} (var {var_ratio[pc]:.3f}):  C4 {a4:+.3f}   C5 {a5:+.3f}')


tri_1 (n=1098):
  PC1 (var 0.547):  C4 -0.980   C5 +0.260
  PC2 (var 0.161):  C4 +0.139   C5 -0.086
  PC3 (var 0.108):  C4 +0.036   C5 -0.072

tri_2 (n=1094):
  PC1 (var 0.536):  C4 -0.978   C5 +0.159
  PC2 (var 0.155):  C4 -0.019   C5 -0.047
  PC3 (var 0.098):  C4 -0.016   C5 +0.157

tri_0 (n=792):
  PC1 (var 0.541):  C4 -0.985   C5 +0.314
  PC2 (var 0.156):  C4 +0.001   C5 -0.006
  PC3 (var 0.101):  C4 -0.106   C5 +0.116

tri_3 (n=613):
  PC1 (var 0.541):  C4 -0.981   C5 -0.058
  PC2 (var 0.145):  C4 -0.044   C5 +0.051
  PC3 (var 0.093):  C4 -0.053   C5 -0.013


In [9]:
# Second layer of the onion: joint strata from the count arrays directly.
# Strata shrink fast — report everything above threshold, expect few.

MIN_STRATUM_L2 = 12

joint = {}
for k in keys:
    joint.setdefault((num_triangles[k], num_4_cycles[k]), []).append(k)

l2_pvals = []
print(f'{"(tri,C4)":>10}   n     C5 rho     p')
for (t, q), members in sorted(joint.items()):
    if len(members) < MIN_STRATUM_L2:
        continue
    idx = np.array(members)
    rho5, p5 = stratum_test(idx, num_5_cycles)
    if rho5 is None:
        continue
    l2_pvals.append(p5)
    print(f'  ({t:2d},{q:2d})   {len(idx):3d}   {rho5:+.3f}   {p5:.4f}')

if l2_pvals:
    p_global, k = fisher_combine(l2_pvals)
    print(f'\nFisher combined across {k} joint strata:  C5 p = {p_global:.2e}')
else:
    print('\nNo joint strata above threshold — increase n or lower MIN_STRATUM_L2.')

  (tri,C4)   n     C5 rho     p


  0%|          | 0/10000 [00:00<?, ?it/s]

  ( 0, 0)    49   +0.974   0.0001


  0%|          | 0/10000 [00:00<?, ?it/s]

  ( 0, 1)   107   +0.950   0.0001


...

  ( 5, 3)    29   +0.428   0.0001


  0%|          | 0/10000 [00:00<?, ?it/s]

  ( 5, 5)    12   +0.193   0.1463


  0%|          | 0/10000 [00:00<?, ?it/s]

  ( 6, 3)    14   +0.368   0.0040

Fisher combined across 43 joint strata:  C5 p = 2.31e-101
